# 02 - DATA CLEANING
#### El objetivo de este notebook es la limpieza y transformación de los datos para utilizarlos en el entrenamientos de modelo.

###

### 1. Importamos las funciones y librerias necesarias.

In [ ]:
import sys
import pandas as pd
import numpy as np

sys.path.append("..")

from src.config import RAW_DATA_PATH, PROCESSED_DATA_PATH

from src.data_loader import load_csv
from src.cleaning import parse_rating, parse_reviews, parse_price, parse_coupon_type
from src.cleaning import parse_bought_last_month, parse_coupon_value


### 2. Cargamos el dataset sucio.

In [64]:
df = load_csv(RAW_DATA_PATH)
df.head()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


### 3. Copiamos el dataframe original para trabajar con la copia.

In [65]:
df_clean = df.copy()


### 3. Modificamos el nombre de la columna `current/discounted_price` para mayor comodidad y eliminamos las columnas que no vamos a utilizar en este proyecto.

In [66]:
df_clean = df_clean.rename(columns={
    "current/discounted_price": "current_discounted_price"
})

columns_to_drop = ["product_url", "image_url"]
df_clean = df_clean.drop(columns=columns_to_drop)

df_clean.head()

,title,rating,number_of_reviews,bought_in_last_month,current_discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,2025-08-21 11:14:29


### 4. Con las funciones creadas, modificamos las columnas necesarias a tipo numérico.

In [ ]:
df_clean["rating"] = df_clean["rating"].apply(parse_rating)

df_clean["number_of_reviews"] = df_clean["number_of_reviews"].apply(parse_reviews)

df_clean["current_discounted_price"] = df_clean["current_discounted_price"].apply(parse_price)

df_clean["listed_price"] = df_clean["listed_price"].apply(parse_price)

# En este caso, solo queremos que parsee las filas dónde aparezca el dolar para evitar coger días u otro números que no nos interesan. 
df_clean["price_on_variant"] = df_clean["price_on_variant"].apply(
    lambda x: parse_price(x, require_dollar=True))

df_clean["bought_in_last_month"] = df_clean["bought_in_last_month"].apply(parse_bought_last_month)

### 5. Creamos dos columnas nuevas a partir de `is_cuponed` indicando la cantidad y el tipo de descuento.

In [68]:
df_clean["coupon_discount_value"] = df_clean["is_couponed"].apply(parse_coupon_value)

df_clean["coupon_discount_type"] = df_clean["is_couponed"].apply(parse_coupon_type)

### 6. Modificamos las columnas necesarias a tipo boolean.

In [69]:
df_clean["is_sponsored"] = (df_clean["is_sponsored"] == "Sponsored").astype(int)

df_clean["is_best_seller"] = (df_clean["is_best_seller"] == "Best Seller").astype(int)

coupon_text = df["is_couponed"].fillna("").astype(str).str.strip().str.lower()
df_clean["is_couponed"] = coupon_text.str.contains("with coupon").astype(int)

df_clean["buy_box_availability"] = (df_clean["buy_box_availability"] == "Add to cart").astype(int)

df_clean["sustainability_badges"] = (df_clean["sustainability_badges"].notna()).astype(int)

df_clean["delivery_details"] = df["delivery_details"].notna().astype(int)

### 7. Creamos nuevas columnas con la cantidad y porcentaje de descuento, y si hubiera algún valor negativo sustituimos su valor por 0.

In [70]:
df_clean["discount_amount"] = (df_clean["listed_price"] - df_clean["current_discounted_price"])

df_clean.loc[df_clean["discount_amount"] < 0, "discount_amount"] = np.nan

df_clean["discount_percentage"] = (df_clean["discount_amount"] / df_clean["listed_price"] * 100).round(2)

### 8. Formateamos la fecha y creamos columnas con la longitud del título en caracteres y palabras.

In [71]:
df_clean["collected_at"] = pd.to_datetime(df_clean["collected_at"], errors="coerce")

df_clean["title_length"] = df_clean["title"].astype(str).str.len()

df_clean["title_word_count"] = df_clean["title"].astype(str).str.split().str.len()

### 9. Creamos nuevas columnas a partir de `delivery_details`.

In [72]:
df_clean["has_free_delivery"] = (
    df["delivery_details"]
    .fillna("")
    .str.lower()
    .str.contains("free delivery")
).astype(int)

df_clean["has_fastest_delivery"] = (
    df["delivery_details"]
    .fillna("")
    .str.lower()
    .str.contains("fastest delivery")
).astype(int)

df_clean = df_clean.drop(columns=["delivery_details"])

### 10. Calculamos el número de nulos.

In [73]:
df_clean.isna().sum().sort_values(ascending=True)

title                           0
is_best_seller                  0
coupon_discount_type            0
collected_at                    0
sustainability_badges           0
buy_box_availability            0
is_couponed                     0
is_sponsored                    0
has_fastest_delivery            0
has_free_delivery               0
title_word_count                0
title_length                    0
rating                       1024
number_of_reviews            1024
bought_in_last_month        10511
current_discounted_price    11749
price_on_variant            22604
discount_percentage         30364
listed_price                30364
discount_amount             30364
coupon_discount_value       40727
dtype: int64

### 11. Revisamos la información del dataset.

In [74]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     42675 non-null  str           
 1   rating                    41651 non-null  float64       
 2   number_of_reviews         41651 non-null  float64       
 3   bought_in_last_month      32164 non-null  float64       
 4   current_discounted_price  30926 non-null  float64       
 5   price_on_variant          20071 non-null  float64       
 6   listed_price              12311 non-null  float64       
 7   is_best_seller            42675 non-null  int64         
 8   is_sponsored              42675 non-null  int64         
 9   is_couponed               42675 non-null  int64         
 10  buy_box_availability      42675 non-null  int64         
 11  sustainability_badges     42675 non-null  int64         
 12  collected_at              426

### 12. Guardamos el dataset.

In [75]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

In [76]:
df_clean.head()

,title,rating,number_of_reviews,bought_in_last_month,current_discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,...,sustainability_badges,collected_at,coupon_discount_value,coupon_discount_type,discount_amount,discount_percentage,title_length,title_word_count,has_free_delivery,has_fastest_delivery
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300.0,89.68,NaN,159.00,0,1,1,...,1,2025-08-21 11:14:29,15.0,percentage,69.32,43.60,199,30,0,0
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000.0,9.99,NaN,15.99,0,1,0,...,0,2025-08-21 11:14:29,NaN,no_coupon,6.00,37.52,200,34,0,0
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000.0,314.00,NaN,349.00,0,1,0,...,0,2025-08-21 11:14:29,NaN,no_coupon,35.00,10.03,193,29,0,0
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882.0,10000.0,NaN,162.24,NaN,1,0,0,...,0,2025-08-21 11:14:29,NaN,no_coupon,NaN,NaN,194,24,0,0
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988.0,10000.0,NaN,72.74,NaN,0,0,0,...,0,2025-08-21 11:14:29,NaN,no_coupon,NaN,NaN,133,24,0,0


### Conclusión de la limpieza de datos

#### En este notebook, se ha realizado la limpieza inicial del dataset, partiendo del archivo original y generando una versión preparada para el análisis exploratorio.

#### Todas las columnas venían en tipo texto, por lo que se han transformado las columnas necesarias tanto en tipo numérico como en tipo binario. También, se han creado nuevas columnas que utilizaremos más adelante.

#### Se puede observar que las columnas siguen teniendo valores nulos. No se han eliminado todavía porque representan ausencia real de información. Se analizará y se tomarán las decisiones correspondientes en el EDA.

#### Por último, se ha guardado el dataset limpio.
